In [1]:
from platform import python_version
print(python_version())

3.11.14


In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as npmtd
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

import json
import requests
import pandas as pd

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.GDC_lib import GDC
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/config/all_lfc_cutoffs_TCGA-PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=False, verbose=False)
# print("\nEcho Parameters:")
# print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD
>>> Tumor


### GDC - no memory restriction to get all data available

In [5]:
gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [6]:
force=False
verbose=False

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)
prog_list.sort()

"; ".join(prog_list)

'ALCHEMIST; APOLLO; BEATAML1.0; CCDI; CCG; CDDP_EAGLE; CGCI; CMI; CPTAC; CTSP; EXCEPTIONAL_RESPONDERS; FM; HCMI; MATCH; MMRF; MP2PRT; NCICCR; OHSU; ORGANOID; RC; REBC; TARGET; TCGA; TRIO; VAREPOP; WCDT'

In [7]:
PROG_ID = 'TCGA'

df_psi = gdc.get_primary_sites(prog_id=PROG_ID, verbose=verbose)
print(len(df_psi))
df_psi.head(3)

33


,prog_id,gdc_project_id,disease_id,psi_id,primary_site,disease_context,cbioportal_study_id,mapping_status,recommended_mutation_source,notes,alternative_cbioportal_study_ids,review_notes
0,TCGA,TCGA-ACC,ACC,TCGA-ACC,Adrenal gland,Adrenocortical carcinoma,acc_tcga_pan_can_atlas_2018,formulaic_tcga_pan_can_atlas,GDC MAF first; cBioPortal only if mapping is validated,Validate with cBioPortal /api/studies before production.,NaN,NaN
1,TCGA,TCGA-BLCA,BLCA,TCGA-BLCA,Bladder,Bladder urothelial carcinoma,blca_tcga_pan_can_atlas_2018,formulaic_tcga_pan_can_atlas,GDC MAF first; cBioPortal only if mapping is validated,Validate with cBioPortal /api/studies before production.,NaN,NaN
2,TCGA,TCGA-BRCA,BRCA,TCGA-BRCA,Breast,Breast invasive carcinoma,brca_tcga_pan_can_atlas_2018,formulaic_tcga_pan_can_atlas,GDC MAF first; cBioPortal only if mapping is validated,Validate with cBioPortal /api/studies before production.,NaN,NaN


### Open primary cites from cbio

In [8]:
df_psi = gdc.open_primary_sites_cbio(verbose=True)

df_psi.columns

Table opened ((89, 12)) at '/home/flavio/uv/perturb_agent/data/gdc_to_cbioportal_study_mapping.tsv'


Index(['prog_id', 'gdc_project_id', 'disease_id', 'psi_id', 'primary_site', 'disease_context',
       'cbioportal_study_id', 'mapping_status', 'recommended_mutation_source', 'notes',
       'alternative_cbioportal_study_ids', 'review_notes'],
      dtype='object')

In [9]:
df_psi.head(3).T

,0,1,2
prog_id,TCGA,TCGA,TCGA
gdc_project_id,TCGA-ACC,TCGA-BLCA,TCGA-BRCA
disease_id,ACC,BLCA,BRCA
psi_id,TCGA-ACC,TCGA-BLCA,TCGA-BRCA
primary_site,Adrenal gland,Bladder,Breast
disease_context,Adrenocortical carcinoma,Bladder urothelial carcinoma,Breast invasive carcinoma
cbioportal_study_id,acc_tcga_pan_can_atlas_2018,blca_tcga_pan_can_atlas_2018,brca_tcga_pan_can_atlas_2018
mapping_status,formulaic_tcga_pan_can_atlas,formulaic_tcga_pan_can_atlas,formulaic_tcga_pan_can_atlas
recommended_mutation_source,GDC MAF first; cBioPortal only if mapping is validated,GDC MAF first; cBioPortal only if mapping is validated,GDC MAF first; cBioPortal only if mapping is validated
notes,Validate with cBioPortal /api/studies before production.,Validate with cBioPortal /api/studies before production.,Validate with cBioPortal /api/studies before production.


In [10]:
DISEASE_ID = 'ACC'
DISEASE_ID = 'PAAD'

df_psi = df_psi[ (df_psi.disease_id == DISEASE_ID) & (~pd.isnull(df_psi.primary_site)) & (~pd.isnull(df_psi.cbioportal_study_id)) ].copy()
dfa = df_psi.groupby(['prog_id', 'psi_id', 'disease_id', 'primary_site', 'gdc_project_id', 'cbioportal_study_id']).size().reset_index()

dfa[ ['prog_id', 'psi_id', 'disease_id', 'primary_site', 'gdc_project_id', 'cbioportal_study_id'] ]

,prog_id,psi_id,disease_id,primary_site,gdc_project_id,cbioportal_study_id
0,CCLE,CCLE-PAAD,PAAD,Pancreas,CCLE-PAAD,ccle_broad_2019
1,CPTAC3,PAAD,PAAD,Pancreas,CPTAC-3,paad_cptac_2021
2,CPTAC3,PAAD_GDC-1,PAAD,Pancreas,CPTAC-3,pancreas_cptac_gdc
3,CPTAC3,PAAD_GDC-2,PAAD,Pancreas,CPTAC-3,pancreas_cptac_gdc
4,TCGA,TCGA-PAAD,PAAD,Pancreas,TCGA-PAAD,paad_tcga_pan_can_atlas_2018


In [11]:
prog_id = 'CCLE'  # DepMap 
psi_id = 'CCLE-BRCA'

prog_id = 'CPTAC'
psi_id = 'CPTAC-BRCA'

prog_id = 'TCGA'
psi_id = 'TCGA-ACC'
psi_id = 'TCGA-BRCA'
psi_id = 'TCGA-BLCA'

prog_id = 'CPTAC3'
psi_id = 'PAAD'


df_psi = gdc.get_primary_sites(prog_id=prog_id, verbose=verbose)
df_psi.head(2)

,prog_id,gdc_project_id,disease_id,psi_id,primary_site,disease_context,cbioportal_study_id,mapping_status,recommended_mutation_source,notes,alternative_cbioportal_study_ids,review_notes
0,CPTAC3,CPTAC-3,BRAIN,BRAIN,Brain,Pediatric brain cancer,brain_cptac_2020,candidate_validate,GDC MAF first; cBioPortal only if mapping is validated,Check exact primary_site/diagnosis before mapping.,NaN,NaN
1,CPTAC3,CPTAC-3,BRCA,BRCA,Breast,Breast cancer context,brca_cptac_2020,reviewed_candidate_cptac,GDC MAF first; cBioPortal only if mapping is validated,Validate cBioPortal study.,breast_cptac_gdc,cBioPortal has BRCA CPTAC 2020 and Breast CPTAC GDC 2025; validate which mat...


In [12]:

gdc.set_primary_site(psi_id)

True

In [13]:
gdc.gdc_project_id, gdc.primary_site

('CPTAC-3', 'Pancreas')

In [ ]:
force=False
verbose=True

df_cases = pdreadcsv(gdc.fname_cases, gdc.root_disease, verbose=True )

print(df_cases.shape)
print(df_cases.primary_site.unique())
df_cases.head(3)

..

👉 Returned 170 / Total paginated 170
Table saved ((170, 26)) at '/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/cases_for_PAAD.tsv'
Table saved ((10, 6)) at '/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/subtype_for_PAAD.tsv'

Could not find clinical demographics data.
(0, 26)
[]


,primary_site,disease_type,case_id,diagnoses,gdc_project_id,subtype_global,stage_ajcc,primary_diagnosis,tumor_grade,stage_clin,...,primary_site_norm,disease_type_norm,diagnosis_norm,tumor_class,histology,subtype_tissue,consistency,validity,n,frac


In [16]:
len(df_cases.case_id.unique())

0

In [ ]:
fname_all_cases = gdc.fname_all_cases % (gdc.prog_id)
filename_cases = gdc.root_summary / fname_all_cases

df_all_cases = pdreadcsv(fname_all_cases, gdc.root_summary, verbose=True)

print(df_all_cases.shape)

len(df_cases.case_id.unique())

Table opened ((1524, 26)) at '/home/flavio/uv/perturb_agent/data/CPTAC3/summary/CPTAC3_summ_cases.tsv'
(1524, 26)


0

In [ ]:
gdc.df_clin_demo

In [ ]:
df_clin_demo = gdc.get_gdc_clin_demo_data(df_cases.case_id, force=True, batch_size=10, verbose=True)
df_clin_demo

In [ ]:
df_subt

In [ ]:
for isubt, row in df_subt.iterrows():
    subtype_global = row.subtype_global
    tumor_class = row.tumor_class
    subtype_tissue = row.subtype_tissue

    df_samples = gdc.get_samples_for_subtypes(
        subtype_global=subtype_global,
        tumor_class=tumor_class,
        subtype_tissue=subtype_tissue,
        batch_size=200,
        force=False,
        verbose=verbose,
    )
    print(f"{isubt}) {gdc.s_case}")

    if df_samples.empty:
        print(f"No samples found for PSI_ID: {psi_id} subtype: {subtype_global} tumor_class: {tumor_class} subtype_tissue: {subtype_tissue}")
    else:
        print(f"There are {len(df_samples)} samples for PSI_ID: {psi_id} subtype: {subtype_global} tumor_class: {tumor_class} subtype_tissue: {subtype_tissue}")
